# ECOMDESIGN — thermodynamically informed design of microbial consortia

Reproduces every result and figure of the manuscript. Run top to bottom.

* **Layer 1 (metabolism):** strain library from a genome-scale model, thermodynamics-based FBA (TFA) with a shared extracellular pool, Max-min Driving Force.
* **Layer 2 (pair features):** competition $C_{ij}$, stoichiometric complementarity $K_{ij}$, thermodynamic exchange window $G_{ij}$.
* **Layer 3 (statistics):** Fermi–Dirac dyadic model $P(y_{ij}=1)=[1+e^{(\eta_{ij}-\mu)/T}]^{-1}$, dyadic-robust inference, bias correction, REM diagnostics.
* **Layer 4 (design):** energy-ranked consortia validated with multi-member TFA.

Heavy steps are cached in `data/`. Set `REBUILD = True` to recompute them (≈1 h for N = 80 on two cores).

In [ ]:
# %pip install cobra equilibrator-api scikit-learn statsmodels  # first run only
import pickle, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from ecomdesign.metabolism import load_base, set_ext_ceiling, CARBONS
from ecomdesign.library import (design_medium, build_library, node_table,
                                exchange_windows, pair_features, pair_truth)
from ecomdesign import analysis as A, stats as S, figures as F

REBUILD = False
SEED, N = 11, 80
CEILINGS = [1e-2, 1e-3, 1e-4, 1e-5]   # mol/L, ceiling of extracellular by-products
Path("data").mkdir(exist_ok=True); Path("figures").mkdir(exist_ok=True)
base = load_base(); D = design_medium(base)

## 1. Standard Gibbs energies
`ecomdesign/dg0_ecoli_core.json` holds $\Delta_r G'^{\circ}$ (pH 7, I = 0.25 M) from eQuilibrator 3.0 for every reaction of the *E. coli* core model. The cell below regenerates it.

In [ ]:
if REBUILD:
    from cobra.io import load_model
    from equilibrator_api import ComponentContribution, Q_, Reaction
    import json
    m = load_model("textbook"); cc = ComponentContribution()
    cc.p_h = Q_(7.0); cc.ionic_strength = Q_("0.25M")
    dg = {}
    for r in m.reactions:
        if r.boundary or r.id.startswith("Biomass"): continue
        st = {}
        for met, c in r.metabolites.items():
            k = met.id.rsplit("_", 1)[0]; st[k] = st.get(k, 0) + c
        st = {k: v for k, v in st.items() if abs(v) > 1e-9}
        if not st: dg[r.id] = (0.0, 0.0); continue
        rx = Reaction({cc.get_compound("bigg.metabolite:" + k): v for k, v in st.items()})
        g = cc.standard_dg_prime(rx)
        dg[r.id] = (float(g.value.m_as("kJ/mol")), float(g.error.m_as("kJ/mol")))
    json.dump(dg, open("ecomdesign/dg0_ecoli_core.json", "w"), indent=1)

## 2. Library, exchange windows and mechanistic ground truth

In [ ]:
lib_file = Path(f"data/library_seed{SEED}_N{N}.pkl")
if REBUILD or not lib_file.exists():
    set_ext_ceiling(1e-2)
    lib = build_library(base, n=N, seed=SEED)
    nodes, Ssec = node_table(base, lib, D)
    LIB = {"lib": lib, "nodes": nodes, "Ssec": Ssec, "res": {}}
    for c in CEILINGS:
        set_ext_ceiling(c)
        lmax, lmin = exchange_windows(base, lib, nodes, D)
        Fp = pair_features(base, lib, nodes, Ssec, lmax, lmin, D)
        Y = pair_truth(base, lib, nodes, D, thermo_only=(c != CEILINGS[0]))
        df = Fp.merge(Y, on=["i", "j"])
        if c != CEILINGS[0]:
            df["y_stoich"] = LIB["res"][CEILINGS[0]]["pairs"]["y_stoich"].values
        LIB["res"][c] = {"lmax": lmax, "lmin": lmin, "pairs": df}
    pickle.dump(LIB, open(lib_file, "wb"))
LIB = pickle.load(open(lib_file, "rb"))
nodes, lib = LIB["nodes"], LIB["lib"]
print(len(lib), "strains;", int(nodes.grows_D.sum()), "producers")
pd.DataFrame({c: {"pairs": len(r["pairs"]), "FBA-feasible": r["pairs"].y_stoich.sum(),
                  "TFA-feasible": r["pairs"].y.sum()} for c, r in LIB["res"].items()}).T

In [ ]:
F.fig_thermo_filter(LIB["res"], "figures/fig2_thermo_filter.png")
from IPython.display import Image; Image("figures/fig2_thermo_filter.png")

## 3. Benchmark (node-split cross-validation), inference, REM and design validation

In [ ]:
ana_file = Path(f"data/analysis_N{N}.pkl")
if REBUILD or not ana_file.exists():
    import subprocess
    subprocess.run(["python", "run_analysis.py", str(lib_file), f"N{N}", "20", "30"], check=True)
ANA = pickle.load(open(ana_file, "rb"))
summary = pd.concat({c: cv.groupby("model")[["auc", "ap"]].agg(["mean", "std"])
                     for c, cv in ANA["cv"].items()}, names=["ceiling"])
summary.round(3)

In [ ]:
F.fig_benchmark(ANA["cv"], "figures/fig3_benchmark.png"); Image("figures/fig3_benchmark.png")

In [ ]:
# interaction-energy coefficients (beta = -theta, units of T) with dyadic-robust SE
ANA["fit"][(1e-4, "M3: + thermodynamic window")][0].round(3)

In [ ]:
r = ANA["rem"]
F.fig_rem(r["diag"], r["theory"], r["temps"], "figures/fig5_rem.png")
print("sigma =", round(r["diag"]["sigma"], 2), " T_c =", round(r["diag"]["Tc"], 3))
Image("figures/fig5_rem.png")

In [ ]:
F.fig_design(ANA["val"], "figures/fig6_design.png")
display(ANA["val"].groupby(["ceiling", "strategy"]).success.agg(["mean", "count"]))
Image("figures/fig6_design.png")

## 4. Monte Carlo study of the estimator

In [ ]:
mc_file = Path("data/montecarlo.pkl")
if REBUILD or not mc_file.exists():
    import subprocess
    subprocess.run(["python", "run_montecarlo.py", "500", "100"], check=True)
MC = pickle.load(open(mc_file, "rb")); mc = pd.DataFrame(MC["rows"])
F.fig_montecarlo(mc, MC["beta"], "figures/fig4_montecarlo.png")
Image("figures/fig4_montecarlo.png")

## 5. Case study: curated hypersaline community + two introduced taxa

Real genome-scale reconstructions (17 curated resident taxa in one SBML file,
plus iJO1366 and iNJ661). The heavy stages live in `community/run_hypersaline.py`
and are run from the shell:

```
python run_hypersaline.py solo                 # solo growth, FBA/TFA, P in {0,1,3,10}
python run_hypersaline.py pairs 0              # 342 directed pairs at P = 0
python run_hypersaline.py pairs 3              # ... and at P = 3
python run_hypersaline.py invasion fba         # invasion from rare
SELFISH=0.999 TAG=_strict python run_hypersaline.py invasion fba
```

The cells below load the resulting tables and rebuild Figure 7.

In [ ]:
CASE = Path("../community")       # adjust if the case-study folder sits elsewhere
solo = pd.read_csv(CASE / "data_solo.csv")
solo.query("abund == 1").pivot_table(index="taxon", columns=["thermo", "P"], values="mu").round(3)

In [ ]:
# thermodynamics vs stoichiometry on the 342 directed pairs at P = 0
p0 = pd.read_csv(CASE / "data_pairs_P0.csv")
w = p0.pivot_table(index=["i", "j"], columns="thermo", values="mu_j")
print("pairs changed by >0.01 1/h:", int((w[True] - w[False]).abs().gt(0.01).sum()))
print("focal taxon grows (TFA):", round(p0.query("thermo").mu_j.gt(0.01).mean(), 3))

In [ ]:
# osmotic cost of a single taxon, solved on demand
import sys; sys.path.insert(0, ".")
from ecomdesign.hetero import tfa_lazy
import pickle as pk
nets, EXT = pk.load(open(CASE / "nets.pkl", "rb")); D = pk.load(open(CASE / "medium.pkl", "rb"))
for P in (0.0, 3.0, 10.0):
    sol, info = tfa_lazy([nets["Escherichia"]], [1.0], D, EXT, osm_pool=P)
    print("P =", P, " mu =", round(sol["bio"][0], 3), " MDF =", round(info["mdf"], 2))

In [ ]:
import subprocess
subprocess.run([sys.executable, "fig_case.py"], cwd=CASE, check=True)
Image("figures/fig7_hypersaline.png")